# asyncio for AI services

## Part 1 · Functions that wait, and how long they take

In [ ]:
import time

Five functions. Each one stands in for something a service waits on (a model, a database, a search index).
The wait is a `time.sleep`.

In [ ]:
def classify():
    time.sleep(0.1)

def lookup_profile():
    time.sleep(0.2)

def search_docs():
    time.sleep(0.3)

def generate_reply():
    time.sleep(0.4)

def save_result():
    time.sleep(0.5)

To measure how long something takes we need a clock. `time.perf_counter()` returns the current time in seconds;
the difference between two readings is the elapsed time.

In [ ]:
start = time.perf_counter()
classify()
print(f"classify took {time.perf_counter() - start} s")

Now all five, one after another, timing each one and the total.

In [ ]:
steps = [classify, lookup_profile, search_docs, generate_reply, save_result]

total_start = time.perf_counter()

for step in steps:
    start = time.perf_counter()
    step()
    print(f"{step.__name__:<15} {time.perf_counter() - start:2f} s")

print(f"\ntotal           {time.perf_counter() - total_start:.2f} s")

---
## Part 2 · The same functions, async

The same five functions, written as coroutines. Two changes: `async def` instead of `def`, and `await asyncio.sleep` instead of `time.sleep`.

In a notebook you can use `await` directly in a cell. In a `.py` file you'd start the loop with `asyncio.run(...)`.

In [ ]:
import asyncio

async def classify():
    await asyncio.sleep(0.1)

async def lookup_profile():
    await asyncio.sleep(0.2)

async def search_docs():
    await asyncio.sleep(0.3)

async def generate_reply():
    await asyncio.sleep(0.4)

async def save_result():
    await asyncio.sleep(0.5)

Calling an `async def` function does not run it. It gives back a *coroutine object*.

In [ ]:
result = classify()
print(result)

To actually run it, we `await` it.

In [ ]:
start = time.perf_counter()
await classify()
print(f"classify took {time.perf_counter() - start:.2f} s")

Now all five, awaited one after another, with the same timing code as Part 1.

In [35]:
async_steps = [classify, lookup_profile, search_docs, generate_reply, save_result]

total_start = time.perf_counter()

for step in async_steps:
    start = time.perf_counter()
    await step()
    print(f"{step.__name__:<15} {time.perf_counter() - start:.2f} s")

print(f"\ntotal           {time.perf_counter() - total_start:.2f} s")

classify        0.10 s
lookup_profile  0.20 s
search_docs     0.30 s
generate_reply  0.40 s
save_result     0.50 s

total           1.50 s


---
## Part 3 · Running them at the same time: `gather` and `run`

So far, awaiting the functions one after another took 1.5 s, the same as the sync version.
`asyncio.gather` starts several coroutines **together** and waits until all of them are done.

In [37]:
start = time.perf_counter()

await asyncio.gather(classify(),lookup_profile(),search_docs(), generate_reply(), save_result(),)

print(f"total {time.perf_counter() - start:.2f} s")

total 0.50 s


Same five functions, same sleeps: 1.5 s became 0.5 s. The total is now the **longest single wait**, not the sum.
Let's see when each one finishes.

All five started at 0.00 s and finished in order of their wait. While one was waiting, the others were too.

**Running it outside a notebook.** A notebook already has an event loop running, which is why we can `await` in a cell.
In a normal `.py` file there is no loop, so you start one with `asyncio.run(main())`.
If you try `asyncio.run` inside the notebook, it complains:

Here is the same thing as a real script. `%%writefile` saves the cell to a file, and `!python` runs it like you would from a terminal.

In [38]:
%%writefile demo.py
import asyncio
import time

async def classify():
    await asyncio.sleep(0.1)

async def lookup_profile():
    await asyncio.sleep(0.2)

async def search_docs():
    await asyncio.sleep(0.3)

async def generate_reply():
    await asyncio.sleep(0.4)

async def save_result():
    await asyncio.sleep(0.5)

async def main():
    start = time.perf_counter()
    await asyncio.gather(classify(), lookup_profile(), search_docs(), generate_reply(), save_result())
    print(f"total {time.perf_counter() - start:.2f} s")

asyncio.run(main())

Overwriting demo.py


In [ ]:
!python demo.py

### `asyncio.run` vs `await asyncio.gather`

They do different jobs, and you normally use them together.

* **`asyncio.run(main())` starts the event loop.** You call it from ordinary, non-async code (for example the bottom of a `.py` file), usually once per program.
  It creates the loop, runs one coroutine to completion, then shuts the loop down. It is the bridge from the normal world into the async world.
* **`await asyncio.gather(a(), b(), c())` runs several coroutines at the same time.** You call it from inside async code, when a loop is already running.
  It waits until all of them are done and returns their results as a list, in the order you passed them.

*Analogy:* `run` opens the kitchen for the day. `gather` is the chef putting five dishes on the stove at once instead of cooking them one by one.

```python
async def main():                      # everything async lives inside here
    await asyncio.gather(              # gather: five things at the same time
        classify(), lookup_profile(), search_docs(),
        generate_reply(), save_result(),
    )

asyncio.run(main())                    # run: starts the loop that main() needs
```

Take either one away and it breaks. Without `gather`, the steps run one after another (1.5 s instead of 0.5 s).
Without `run`, `main()` only creates a coroutine object and nothing runs, because there is no loop.

| | `asyncio.run(x)` | `await asyncio.gather(x, y, z)` |
|---|---|---|
| Called from | normal (sync) code | inside `async def` (or a notebook cell) |
| Needs a loop already running? | no, it creates one | yes |
| Purpose | start the async world | run several things concurrently |
| Typical use | once, at the program's entry point | wherever you want things to overlap |

In a notebook the loop already exists, so we skip `run` and write `await asyncio.gather(...)` straight in a cell.

---
## Part 4 · Limiting how many run at once: `Semaphore`

`gather` starts everything at once. That's fine for five functions, but what if there were 5,000, or the thing you're calling
(a model API, a database) can only handle a few at a time?

An `asyncio.Semaphore(n)` is a counter of tickets. A task must take a ticket to enter, and only `n` tickets exist.
Everyone else waits at the door, and a ticket is returned when a task leaves.

In [ ]:
gate = asyncio.Semaphore(2)

async def limited(step, gate, t0):
    async with gate:
        print(f"{step.__name__:<15} started   at {time.perf_counter() - t0:.2f} s")
        await step()
        print(f"{step.__name__:<15} finished  at {time.perf_counter() - t0:.2f} s")

t0 = time.perf_counter()
await asyncio.gather(*(limited(step, gate, t0) for step in async_steps))
print(f"\ntotal {time.perf_counter() - t0:.2f} s")

Only two run at a time. As soon as one finishes, the next waiting one takes its place.
All five were started by `gather`, but only two were allowed in.
`async with gate:` takes a ticket and always returns it, even if the task fails.

Notice that nothing was rejected or dropped. The others just waited their turn.

In [ ]:
async def limited_quiet(step, gate):
    async with gate:
        await step()

for n in (1, 2, 3, 5):
    gate = asyncio.Semaphore(n)
    start = time.perf_counter()
    await asyncio.gather(*(limited_quiet(step, gate) for step in async_steps))
    print(f"Semaphore({n}): {time.perf_counter() - start:.2f} s")

The limit is a dial between the two extremes. `Semaphore(1)` is the same as running one after another (1.5 s).
`Semaphore(5)` is the same as unrestricted `gather` (0.5 s).

A semaphore doesn't make anything faster. It slows you down on purpose so you don't overwhelm whatever you're calling.

---
## Part 5 · Starting work without waiting: `create_task`

With `gather` we start several coroutines and wait for all of them. Sometimes we want to **start one and keep going**,
collecting the result later (or never). `asyncio.create_task(...)` hands a coroutine to the event loop to run in the background
and gives back a **task** that we can check on.

Two things we need: a slow function (a model call) and some other work to do in the meantime.

In [ ]:
async def get_answer():
    await asyncio.sleep(1.0)          # slow: a model call
    return "Here is your answer"

async def other_work():
    await asyncio.sleep(0.3)          # something else we need to do

First **without** a task: we wait for the answer, and only then do the other work.

In [ ]:
start = time.perf_counter()
answer = await get_answer()
await other_work()
print(f"without task: {time.perf_counter() - start:.1f} s")

Now **with** a task: the answer is "put in the oven" and cooks in the background while we do the other work.

In [ ]:
start = time.perf_counter()
task = asyncio.create_task(get_answer())     # start it now, don't wait
await other_work()                           # do other things meanwhile
answer = await task                          # go back and pick it up
print(f"with task:    {time.perf_counter() - start:.1f} s")

Two separate actions:

* `create_task(...)` means "start this now, in the background". It doesn't wait.
* `await task` means "I'm done with my other work, now I need the result, so I'll wait for whatever is left".

Both versions use `await`. What differs is **when** the slow thing starts. A task lets it start earlier, so its waiting overlaps with our other work.
Note the parentheses: `create_task(get_answer())`. We pass a coroutine object, not the function.

**A task is different from a plain coroutine.** A coroutine object does nothing until something awaits it. A task starts right away.
The same one second of "other work" shows it:

In [ ]:
# A coroutine object does NOT start until it is awaited
start = time.perf_counter()
coro = get_answer()
await asyncio.sleep(1.0)                        # 1 s of "other work"
await coro
print(f"coroutine: {time.perf_counter() - start:.1f} s")

# A task starts right away
start = time.perf_counter()
task = asyncio.create_task(get_answer())
await asyncio.sleep(1.0)                        # the same 1 s of "other work"
await task
print(f"task:      {time.perf_counter() - start:.1f} s")

The coroutine sat idle during the other work and only started when we awaited it: 1 s + 1 s.
The task was already running, so its wait overlapped the other work: 1 s total.

| | `gather(a(), b())` | `create_task(a())` |
|---|---|---|
| Starts | several at once | one |
| Waits? | yes, until all are done | no, you decide when to `await` it |
| Use it for | "do these together and give me all the results" | "start this, I'll carry on, and check on it later" |

(`gather` turns each coroutine into a task for you.)

### A task and a `gather` on the same event loop

`create_task` and `gather` can be used together. Both hand coroutines to the **event loop**, the scheduler that runs everything.
Whenever one coroutine reaches an `await`, the loop switches to another that is ready.

Here `save_result` becomes a background task, and the other four steps run in a `gather`.
The `traced` helper prints when each step starts and finishes.

In [ ]:
async def traced(step, t0):
    print(f"{time.perf_counter() - t0:.2f} s  {step.__name__} started")
    await step()
    print(f"{time.perf_counter() - t0:.2f} s  {step.__name__} finished")

t0 = time.perf_counter()

# 1. send save_result to the event loop as a background task
task = asyncio.create_task(traced(save_result, t0))
print(f"{time.perf_counter() - t0:.2f} s  task created (the loop has it, but we haven't awaited yet)")

# 2. gather the other four. This is our first await, so the loop now gets control
await asyncio.gather(*(traced(step, t0) for step in async_steps if step is not save_result))
print(f"{time.perf_counter() - t0:.2f} s  gather is done, task.done() = {task.done()}")

# 3. collect the task
await task
print(f"\ntotal {time.perf_counter() - t0:.2f} s")

Three things to notice:

1. **"task created" prints before "save_result started".** `create_task` only hands the coroutine to the event loop.
   It starts running when our code reaches its first `await`, which is the `gather` line.
2. **All five start at 0.00 s.** The loop runs the task and the four `gather` coroutines side by side, switching whenever one is waiting.
3. **The task outlives the `gather`.** At 0.40 s the `gather` is finished but the task is still running (`task.done() = False`), so we `await` it to collect it.

Everything overlapped, so the total is 0.5 s, the longest single wait. One after another, it would have been 0.4 + 0.5 = 0.9 s, even with `gather` alone.

### Do we have to `await` the task?

No. The task runs whether or not we await it. The `await` is how we get the **result** back.

In [ ]:
async def work():
    await asyncio.sleep(0.5)
    print("work finished (nobody awaited it)")

task = asyncio.create_task(work())     # we never await this task
print("we moved on")
await asyncio.sleep(1.0)               # the loop stays alive, so the task finishes anyway

There is a catch: a task only runs as long as the **event loop is alive**. In a script, when `main()` returns, `asyncio.run` shuts the loop down
and cancels whatever is still running:

"work finished" never prints: the program ended first. A web server keeps its loop alive forever, so a task started there finishes on its own, with nobody awaiting it.

So there are two reasons to use a task:

1. **Overlap:** start slow work early and carry on, then `await` it to collect the result.
2. **Fire and leave:** start the work and return, with no `await` at all, and let the result go somewhere else.
   (If the task fails and nobody awaits it, nobody notices the error, so background work needs its own error handling.)

---
## Recap

| Tool | What it does | Where we saw it |
|---|---|---|
| `time.perf_counter()` | measures elapsed time (only differences matter) | Part 1 |
| `async def` / `await` | a function that can wait without blocking; `await` runs it | Part 2 |
| `await asyncio.gather(...)` | runs several coroutines at the same time and waits for all | Part 3 |
| `asyncio.run(main())` | starts the event loop from normal code (notebooks already have one) | Part 3 |
| `asyncio.Semaphore(n)` | lets at most `n` run at once; the rest wait their turn | Part 4 |
| `asyncio.create_task(...)` | starts one coroutine in the background and returns a task | Part 5 |

Key ideas:

* Async does not make a single request faster. It lets waits **overlap**.
* Awaiting coroutines one after another is still sequential. Use `gather` or tasks to overlap them.
* A semaphore doesn't speed anything up. It limits concurrency so you don't overwhelm what you call.
* A task runs whether or not you `await` it, as long as the event loop stays alive.